# Module 5.5: Decoding & Sampling

A trained model doesn't hand you words — it hands you **logits**. This notebook is about the missing last mile: how that raw score vector turns into actual text, and the knobs (`temperature`, `top_k`, `top_p`) that decide whether your model sounds focused and factual or wild and creative.

## The mental model

You just trained a model (Module 5.3). When you run a forward pass on a sequence, the model's final layer (`lm_head`) outputs one **logit** per vocabulary token — a raw, unnormalized score saying "how much do I like this token as the next one?"

Think of the model as a sommelier who, instead of naming *one* wine, slides you a scorecard rating **every** bottle in the cellar. **Decoding** is the act of actually picking a bottle off that scorecard. The model never picks for you — *you* (the decoding strategy) do.

There is no single "correct" pick. The same scorecard can yield a safe, predictable choice or a bold, surprising one depending on the rules you bring. Those rules are what this notebook is about.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# Reproducibility: identical randomness on every run.
torch.manual_seed(0)

device = "cpu"  # tiny model, CPU is plenty
print("Ready.")

## 0. A tiny model to play with

To see *real* (non-uniform) logits we need a model that has learned *something*. We won't train a giant model — we just need one that produces lopsided scorecards. So we quick-train a tiny char-level `GPT` on a short embedded poem for a few hundred steps. This takes a few seconds.

> We deliberately keep it small and lightly trained. It will spell *roughly* like the poem and that's all we need — the point is the **decoding strategies**, not a state-of-the-art model. (If you have a checkpoint in `checkpoints/`, feel free to load it instead; we don't depend on one.)

In [ ]:
from llm_workout.model import GPT

# A short, public-domain nursery rhyme, repeated so the model has enough to chew on.
corpus = (
    "twinkle twinkle little star\n"
    "how i wonder what you are\n"
    "up above the world so high\n"
    "like a diamond in the sky\n"
    "twinkle twinkle little star\n"
    "how i wonder what you are\n"
) * 40

# Build a character-level vocabulary.
chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print(f"Vocab size: {vocab_size} characters -> {''.join(chars)!r}")
print(f"Corpus length: {len(corpus)} characters")

In [ ]:
# A small model: enough capacity to learn the rhyme, small enough to train in seconds.
block_size = 32
model = GPT(
    vocab_size=vocab_size,
    d_model=64,
    num_layers=2,
    num_heads=4,
    hidden_dim=128,
    max_seq_len=block_size,
).to(device)

data = torch.tensor(encode(corpus), dtype=torch.long, device=device)

def get_batch(batch_size=32):
    # Random windows of length block_size; target is the same window shifted by 1.
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

model.train()
for step in range(400):
    x, y = get_batch()
    _, loss, _ = model(x, targets=y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0 or step == 399:
        print(f"step {step:3d} | loss {loss.item():.3f}")

model.eval()
print("\nDone. The model now produces lopsided (non-uniform) logits.")

## 1. Logits → probabilities (one position)

Let's feed the model a prompt and look at the scorecard for the **very next** character.

`forward` returns `(logits, loss, kv_caches)`. The logits have shape `(batch, seq_len, vocab_size)` — one score vector per position. We only care about predicting what comes *after* the prompt, so we slice the **last** position: `logits[:, -1, :]`.

Raw logits aren't probabilities (they can be negative, and they don't sum to 1). We run them through **softmax** to turn them into a proper probability distribution over the vocabulary.

In [ ]:
prompt = "twinkle twinkle little st"
idx = torch.tensor([encode(prompt)], device=device)  # shape (1, len)

with torch.no_grad():
    logits, _, _ = model(idx)

next_logits = logits[:, -1, :]          # (1, vocab) -- only the last position matters
probs = F.softmax(next_logits, dim=-1)  # turn scores into a distribution

print(f"Prompt: {prompt!r}")
print(f"logits shape: {tuple(logits.shape)}  ->  sliced last position: {tuple(next_logits.shape)}\n")

# Show the top few candidates the model is considering.
topv, topi = probs[0].topk(6)
print("Top candidates for the next character:")
for p, i in zip(topv.tolist(), topi.tolist()):
    print(f"  {itos[i]!r:5}  logit={next_logits[0, i].item():6.2f}  prob={p:6.2%}")
print("\n(The model wants an 'a' -- it's spelling 'star'.)")

## 2. Greedy decoding (argmax)

The simplest rule: **always pick the single highest-probability token.** That's just `argmax`.

It's **deterministic** — same prompt, same output, every time. Great when you want one reliable answer.

The downside: it's a tunnel-visioned, plays-it-safe diner who orders the exact same dish every night. It can fall into **repetitive loops** ("the the the") and produces dull, predictable text, because it never explores the second- or third-best option even when that would read better overall.

In [ ]:
def greedy(logits):
    """Pick the argmax token. logits: (1, vocab) -> (1, 1) token id."""
    return logits.argmax(dim=-1, keepdim=True)

# Greedy is deterministic: run it 3 times, get the identical token every time.
for _ in range(3):
    tok = greedy(next_logits)
    print(f"greedy pick: {itos[tok.item()]!r}")

## 3. Pure sampling (multinomial)

Instead of always taking the top pick, **roll a weighted die**: sample a token *in proportion to its probability*. A token with 60% probability is picked ~60% of the time; a 1% token still gets picked ~1% of the time.

This is what `torch.multinomial` does, and it's exactly what the library's `GPT.generate` uses by default.

Result: text is **varied** and non-repetitive — but because *every* token (even garbage ones in the long tail) has a nonzero chance, pure sampling can occasionally "go off the rails" and pick something nonsensical. The strategies in the rest of the notebook are all about taming that tail.

In [ ]:
def sample(logits):
    """Sample one token in proportion to its probability."""
    probs = F.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

# Pure sampling is stochastic: run it several times and watch the pick vary.
picks = [itos[sample(next_logits).item()] for _ in range(12)]
print("12 pure-sampling picks:", picks)
print("\nMostly the high-prob char, but other characters slip in -- that's the variety.")

## 4. Temperature — the single most important knob

Before softmax, **divide the logits by a number `T`** (the *temperature*):

$$\text{probs} = \text{softmax}(\text{logits} / T)$$

- **Low `T` (e.g. 0.5):** logits get *amplified*, so the gaps between them widen. The distribution becomes **peaky** — the model gets more confident and predictable. As `T → 0` this approaches **greedy**.
- **`T = 1`:** the model's natural, untouched distribution.
- **High `T` (e.g. 2.0):** logits get *squashed* toward each other, so the distribution **flattens** out toward uniform — more random, more surprising, more likely to wander.

The analogy: temperature is how much wine you've given the sommelier. Stone sober (low T), they recommend the obvious safe bottle. A few glasses in (high T), they'll enthusiastically suggest the weird natural wine nobody's heard of.

### Key visualization: same logits, three temperatures

Let's freeze a single, fixed logit vector and *watch the distribution change shape* as we turn the temperature knob. This is the picture to burn into memory.

In [ ]:
# A fixed, hand-picked logit vector over 8 imaginary tokens (model-free, fully reproducible).
fixed_logits = torch.tensor([3.0, 2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0])
labels = [f"tok{i}" for i in range(len(fixed_logits))]
temps = [0.5, 1.0, 2.0]
titles = ["T = 0.5  (peaky / confident)", "T = 1.0  (natural)", "T = 2.0  (flat / random)"]
colors = ["#2a9d8f", "#264653", "#e76f51"]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, T, title, color in zip(axes, temps, titles, colors):
    p = F.softmax(fixed_logits / T, dim=-1)
    ax.bar(labels, p.numpy(), color=color)
    ax.set_title(title)
    ax.set_ylim(0, 1.0)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("probability")
fig.suptitle("Same logits, different temperature", fontsize=13)
plt.tight_layout()
plt.show()

print("Left: one token dominates (almost greedy).  Right: probabilities even out (almost a coin flip across all 8).")

### Clearing up the misconceptions

- **"Higher temperature is smarter / more creative-in-a-good-way."** No. Higher temperature just makes the model **less picky**, not more intelligent. It doesn't add new knowledge — it only redistributes probability toward options the model already considered unlikely. Crank it high enough and you get gibberish, not genius.
- **"Why not always use T = 1?"** `T = 1` is the model's raw distribution, which is often *too* loose for tasks that demand precision. You tune T to the job:
  - **`T < 1`** for focused, factual, deterministic-ish work (code, math, extraction, Q&A).
  - **`T > 1`** for brainstorming, poetry, and variety.
  - **`T → 0`** collapses to **greedy** (always the top token).
- **Temperature is a dial, not a switch.** There's a smooth **creativity ↔ consistency** tradeoff, and the sweet spot depends entirely on what you're generating.

In [ ]:
def apply_temperature(logits, T):
    """Scale logits by temperature. T<1 sharpens, T>1 flattens."""
    if T == 0:
        # T->0 is greedy. Build a one-hot on the argmax to avoid divide-by-zero.
        out = torch.full_like(logits, float("-inf"))
        out[..., logits.argmax(dim=-1)] = 0.0
        return out
    return logits / T

## 5. Top-k sampling

Temperature reshapes the *whole* distribution but never zeroes anything out — that long tail of garbage tokens still has a (tiny) chance.

**Top-k** is a hard cutoff: keep only the **`k` highest-logit tokens**, throw the rest away (set them to `-inf`), renormalize, and sample from just those `k`.

Analogy: the sommelier only considers the **top 5 bottles** and refuses to even mention the rest. You still get variety (you sample among the 5), but you can never accidentally get served the bottle of vinegar at the back of the cellar.

In [ ]:
def top_k_filter(logits, k):
    """Keep only the k highest logits; set the rest to -inf so softmax ignores them."""
    if k is None or k >= logits.size(-1):
        return logits
    kth_value = torch.topk(logits, k, dim=-1).values[..., -1, None]  # the k-th largest logit
    return torch.where(logits < kth_value, torch.full_like(logits, float("-inf")), logits)

# Demo on our fixed 8-token vector with k=3: only the top 3 survive.
filtered = top_k_filter(fixed_logits.clone(), k=3)
p = F.softmax(filtered, dim=-1)
print("top-k (k=3) probabilities over the 8 tokens:")
for lab, prob in zip(labels, p.tolist()):
    bar = "#" * int(prob * 40)
    print(f"  {lab}: {prob:6.2%} {bar}")
print("\nOnly 3 tokens have nonzero probability; the tail is gone.")

## 6. Top-p (nucleus) sampling

Top-k uses a *fixed* number of tokens — but that's a blunt instrument. Sometimes the model is super confident (one token deserves 95% — keeping 50 is silly); sometimes it's genuinely unsure (the truth is spread across 80 tokens — keeping only 5 is too aggressive).

**Top-p** (a.k.a. **nucleus sampling**) adapts: sort tokens by probability, then keep the **smallest set whose cumulative probability ≥ `p`** (e.g. 0.9). Drop the rest, renormalize, sample.

- Model very confident → that set is tiny (maybe 1–2 tokens).
- Model unsure → the set automatically grows to include more options.

It's the sommelier who says "I'll consider however many top bottles it takes to cover 90% of what's good tonight" — few on an easy night, more on a hard one. This is why top-p often beats a fixed top-k.

In [ ]:
def top_p_filter(logits, p):
    """Keep the smallest set of tokens whose cumulative prob >= p; set the rest to -inf."""
    if p is None or p >= 1.0:
        return logits
    sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
    cum_probs = F.softmax(sorted_logits, dim=-1).cumsum(dim=-1)
    # Remove tokens once we've already passed the cumulative threshold.
    remove = cum_probs > p
    # Shift right so we always KEEP the token that tipped us over p (the nucleus boundary).
    remove[..., 1:] = remove[..., :-1].clone()
    remove[..., 0] = False
    sorted_logits[remove] = float("-inf")
    # Scatter the masked logits back into the original token order.
    out = torch.empty_like(logits)
    out.scatter_(-1, sorted_idx, sorted_logits)
    return out

# Compare: a CONFIDENT distribution vs an UNSURE one, both under top-p = 0.9.
confident = torch.tensor([6.0, 1.0, 0.5, 0.0, -1.0, -2.0, -3.0, -4.0])
unsure    = torch.tensor([1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3])
for name, lg in [("confident", confident), ("unsure", unsure)]:
    kept = (top_p_filter(lg.clone(), p=0.9) > float("-inf")).sum().item()
    print(f"{name:>10}  ->  top-p(0.9) keeps {kept} of 8 tokens")
print("\nSame p, different set size: that's the adaptivity top-k can't give you.")

## 7. Repetition penalty (brief)

Even with good sampling, models can lock into loops ("star star star..."). A cheap fix: **downweight tokens that have already appeared**. Before sampling, for every token already in the generated sequence, divide its logit (if positive) or multiply it (if negative) by a penalty > 1 — making it less attractive to repeat.

It's a nudge toward novelty, not a hard ban. A penalty of `1.0` does nothing; `1.2` is a common, gentle setting.

In [ ]:
def apply_repetition_penalty(logits, generated_ids, penalty=1.2):
    """Discourage tokens already present in generated_ids (a 1D tensor of token ids)."""
    if penalty == 1.0 or generated_ids.numel() == 0:
        return logits
    for tok in set(generated_ids.tolist()):
        score = logits[0, tok]
        # Positive logits shrink, negative logits grow more negative -> both become less likely.
        logits[0, tok] = score / penalty if score > 0 else score * penalty
    return logits

## 8. Putting it all together

Now the payoff. Here's one decode loop that:
1. calls `model(idx)` to get logits,
2. slices the last position `logits[:, -1, :]`,
3. applies whichever knobs you asked for (repetition penalty → temperature → top-k → top-p),
4. picks a token (greedy or sample), appends it, and repeats.

This is *exactly* the machinery behind API parameters like `temperature` and `top_p`.

In [ ]:
@torch.no_grad()
def generate_with(prompt, max_new_tokens=150, strategy="sample",
                  temperature=1.0, top_k=None, top_p=None, repetition_penalty=1.0):
    """Autoregressive decode loop with selectable strategy. strategy='greedy' or 'sample'."""
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)

    for _ in range(max_new_tokens):
        # Keep context within the model's max length.
        idx_cond = idx[:, -block_size:]
        logits, _, _ = model(idx_cond)
        logits = logits[:, -1, :]  # (1, vocab): only the next-token distribution

        # --- apply the knobs, in order ---
        logits = apply_repetition_penalty(logits, idx[0], repetition_penalty)
        if strategy == "greedy":
            next_id = logits.argmax(dim=-1, keepdim=True)
        else:
            logits = apply_temperature(logits, temperature)
            logits = top_k_filter(logits, top_k)
            logits = top_p_filter(logits, top_p)
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_id], dim=1)

    return decode(idx[0].tolist())

In [ ]:
prompt = "twinkle "

torch.manual_seed(0)
g_greedy = generate_with(prompt, strategy="greedy")
torch.manual_seed(0)
g_balanced = generate_with(prompt, strategy="sample", temperature=0.7, top_p=0.9)
torch.manual_seed(0)
g_wild = generate_with(prompt, strategy="sample", temperature=1.5)

def show(title, text):
    print(f"--- {title} ---")
    print(repr(text))
    print()

show("GREEDY (deterministic, prone to loops)", g_greedy)
show("T=0.7 + top-p=0.9 (the usual sweet spot: coherent but not robotic)", g_balanced)
show("T=1.5 (high temperature: creative, but starts to drift)", g_wild)

Look at the three outputs side by side. **Greedy** repeats itself the most (it always takes the safe bet). **T=0.7 + top-p** stays on-topic but reads more naturally. **T=1.5** is the most adventurous — and the most likely to wander into nonsense. There's the creativity-vs-consistency tradeoff, in your own model.

> Our model is tiny and only lightly trained on a nursery rhyme, so even the "good" output is rough. On a real model these same knobs are the difference between a crisp factual answer and a rambling one.

## Summary

A model only ever gives you **logits**. Decoding is how you turn that scorecard into text:

| Strategy | What it does | When to reach for it |
|---|---|---|
| **Greedy** | always the top token | deterministic, factual one-shot answers |
| **Pure sampling** | weighted die over all tokens | maximum variety (risky alone) |
| **Temperature** | scale logits by `T` before softmax | the master creativity↔consistency dial |
| **Top-k** | keep the `k` best, drop the rest | cut the garbage tail (fixed size) |
| **Top-p** | keep the smallest set summing to `p` | cut the tail *adaptively* |
| **Repetition penalty** | downweight seen tokens | break out of loops |

When you call an LLM API and set `temperature=0.7` or `top_p=0.9`, **this is the exact code running on the other side.** You now know what every one of those knobs does.

### 🏋️ Try it yourself

1. **Find greedy by another name.** Call `generate_with(prompt, strategy='sample', temperature=0.0, ...)` (it routes through `apply_temperature`'s `T==0` branch) and confirm the output matches `strategy='greedy'`. Temperature 0 *is* greedy.
2. **Sweep the temperature.** Loop `temperature` over `[0.2, 0.7, 1.0, 1.5, 2.0]` (fix the seed each time) and eyeball where the output flips from "too repetitive" to "too chaotic."
3. **Top-k vs top-p.** Generate with `top_k=3` and with `top_p=0.9` from the same prompt+seed. Do they pick the same tokens? When would you prefer one over the other?
4. **Loop-breaker.** Run pure greedy until it gets stuck repeating, then add `repetition_penalty=1.3` and watch the loop break.

In [ ]:
# Your turn -- starter code.
p = "how i wonder "

# Task 1: temperature 0 should equal greedy.
torch.manual_seed(0); a = generate_with(p, max_new_tokens=60, strategy="sample", temperature=0.0)
torch.manual_seed(0); b = generate_with(p, max_new_tokens=60, strategy="greedy")
print("T=0 matches greedy?", a == b)

# Task 2: sweep temperature (fill in and run).
for T in [0.2, 0.7, 1.0, 1.5, 2.0]:
    torch.manual_seed(0)
    out = generate_with(p, max_new_tokens=60, strategy="sample", temperature=T)
    print(f"\nT={T}: {out!r}")